In [1]:
import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords

In [2]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [3]:
stop_words = set(stopwords.words("english"))

print("Total stopwords:", len(stop_words))

Total stopwords: 198


In [4]:
fake = pd.read_csv("../data/raw/Fake.csv")
real = pd.read_csv("../data/raw/Real.csv")

print("Fake dataset:", fake.shape)
print("Real dataset:", real.shape)

Fake dataset: (23481, 4)
Real dataset: (21417, 4)


In [5]:
fake["label"] = 0
real["label"] = 1

In [ ]:
# Combine the datasets
data = pd.concat(
    [fake, real],
    ignore_index=True
)

print("Combined dataset:", data.shape)

Combined dataset: (44898, 5)


In [ ]:
# Handle missing text
data["title"] = data["title"].fillna("")
data["text"] = data["text"].fillna("")

In [8]:
# Remove duplicates
print("Duplicates before:", data.duplicated().sum())

data = data.drop_duplicates().reset_index(drop=True)

print("Duplicates after:", data.duplicated().sum())
print("Dataset shape:", data.shape)

Duplicates before: 209
Duplicates after: 0
Dataset shape: (44689, 5)


In [9]:
# Combine title and article text
data["content"] = data["title"] + " " + data["text"]

In [10]:
print(data["content"].iloc[0])

 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a great year for America! As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year. 2018 will be a great year for America!  Donald J. Trump (@realDonaldTrump) December 31, 2017Trump s tweet went down about as welll as you d expect.What kind of president sends a New Year s greeting like this d

In [11]:
# Create the text cleaning function
def clean_text(text):
    
    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Remove numbers
    text = re.sub(r"\d+", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [12]:
# Apply cleaning
data["clean_content"] = data["content"].apply(clean_text)

In [13]:
print("ORIGINAL:")
print(data["content"].iloc[0])

print("\n" + "=" * 80)

print("CLEANED:")
print(data["clean_content"].iloc[0])

ORIGINAL:
 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a great year for America! As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year. 2018 will be a great year for America!  Donald J. Trump (@realDonaldTrump) December 31, 2017Trump s tweet went down about as welll as you d expect.What kind of president sends a New Year s greeting l

In [ ]:
# Remove stopwords(the, is, a, an, of, to, in)

def remove_stopwords(text):
    
    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)

In [15]:
data["clean_content"] = data["clean_content"].apply(
    remove_stopwords
)

In [16]:
print(data["clean_content"].iloc[0])

donald trump sends embarrassing new year’s eve message disturbing donald trump wish americans happy new year leave instead give shout enemies haters dishonest fake news media former reality show star one job country rapidly grows stronger smarter want wish friends supporters enemies haters even dishonest fake news media happy healthy new year president angry pants tweeted great year america country rapidly grows stronger smarter want wish friends supporters enemies haters even dishonest fake news media happy healthy new year great year america donald j trump realdonaldtrump december trump tweet went welll expectwhat kind president sends new year greeting like despicable petty infantile gibberish trump lack decency even allow rise gutter long enough wish american citizens happy new year bishop talbert swan talbertswan december one likes calvin calvinstowell december impeachment would make great year america also accept regaining control congress miranda yaver mirandayaver december hear 

We'll use simple word-based tokenization:

In [17]:
# Tokenization
def tokenize(text):
    return text.split()

In [18]:
data["tokens"] = data["clean_content"].apply(tokenize)

In [19]:
print(data["tokens"].iloc[0][:30])

['donald', 'trump', 'sends', 'embarrassing', 'new', 'year’s', 'eve', 'message', 'disturbing', 'donald', 'trump', 'wish', 'americans', 'happy', 'new', 'year', 'leave', 'instead', 'give', 'shout', 'enemies', 'haters', 'dishonest', 'fake', 'news', 'media', 'former', 'reality', 'show', 'star']


In [20]:
# Check the processed data
print(data[
    ["title", "clean_content", "tokens", "label"]
].head())

                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                       clean_content  \
0  donald trump sends embarrassing new year’s eve...   
1  drunk bragging trump staffer started russian c...   
2  sheriff david clarke becomes internet joke thr...   
3  trump obsessed even obama’s name coded website...   
4  pope francis called donald trump christmas spe...   

                                              tokens  label  
0  [donald, trump, sends, embarrassing, new, year...      0  
1  [drunk, bragging, trump, staffer, started, rus...      0  
2  [sheriff, david, clarke, becomes, internet, jo...      0  
3  [trump, obsessed, even, obama’s, name, coded, ...      0  
4  [pope, franci

In [21]:
# Check final dataset
print("Final shape:", data.shape)

print("\nMissing values:")
print(data["clean_content"].isnull().sum())

print("\nLabels:")
print(data["label"].value_counts())

Final shape: (44689, 8)

Missing values:
0

Labels:
label
0    23478
1    21211
Name: count, dtype: int64


In [22]:
# Create the dataset for ML

processed_data = data[
    ["clean_content", "label"]
].copy()

In [23]:
processed_data.head()

,clean_content,label
0,donald trump sends embarrassing new year’s eve...,0
1,drunk bragging trump staffer started russian c...,0
2,sheriff david clarke becomes internet joke thr...,0
3,trump obsessed even obama’s name coded website...,0
4,pope francis called donald trump christmas spe...,0


In [24]:
processed_data.to_csv(
    "../data/processed/cleaned_news.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
